### Load Dataset

In [2]:
import pandas as pd

# Agar train.csv aur test.csv same folder me hain
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

train.shape, test.shape

((1460, 81), (1459, 80))

### Exploratory Data Analysis (EDA)

In [3]:
train.head()
train.describe()
train.isnull().sum()

Id                 0
MSSubClass         0
MSZoning           0
LotFrontage      259
LotArea            0
                ... 
MoSold             0
YrSold             0
SaleType           0
SaleCondition      0
SalePrice          0
Length: 81, dtype: int64

### Select Features & Target

In [4]:
target = train["SalePrice"]

In [5]:
features = train.drop(columns=["SalePrice"])

In [15]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor

# Load dataset
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Target
y = train["SalePrice"]

# Features
X = train.drop(columns=["SalePrice"])

# 1️⃣ Numeric columns
num_cols = X.select_dtypes(include=["int64","float64"]).columns
num_imputer = SimpleImputer(strategy="median")
X[num_cols] = num_imputer.fit_transform(X[num_cols])
test[num_cols] = num_imputer.transform(test[num_cols])

# 2️⃣ Categorical columns
cat_cols = X.select_dtypes(include=["object"]).columns
cat_imputer = SimpleImputer(strategy="most_frequent")
X[cat_cols] = cat_imputer.fit_transform(X[cat_cols])
test[cat_cols] = cat_imputer.transform(test[cat_cols])

# 3️⃣ Encode categorical columns (One-Hot)
X = pd.get_dummies(X)
test_encoded = pd.get_dummies(test.drop(columns=["Id"], errors="ignore"))

# Align columns (make sure train & test have same columns)
X, test_encoded = X.align(test_encoded, join="left", axis=1, fill_value=0)

# 4️⃣ Train model
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X, y)

# 5️⃣ Predict
predictions = model.predict(test_encoded)

# 6️⃣ Submission
submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": predictions
})
submission.to_csv("submission.csv", index=False)